# تست augmentation نسخه‌ی ۲: permutation کامل محور سوپرسل (نه فقط سلول مرجع)

## چرا این نسخه؟
نسخه‌ی قبلی فقط محور سلول واحد (۸ اتم مرجع) را permute کرد و محور سوپرسل (۷۲ اتم) را بدون
تغییر گذاشت — این از پایه اشتباه است، چون وقتی تقارن اتم مرجع `i` را به `perm[i]` می‌برد،
باید **همان تقارن** روی موقعیت تمام اتم‌های سوپرسل هم اعمال شود تا بفهمیم هر تصویر `j`
به کدام تصویر دیگر منتقل می‌شود. نتیجه‌ی آن نسخه: اختلاف فرکانس ۱۲.۷۷ THz (کاملاً نادرست).

## رفع باگ
این‌بار permutation را روی **تمام ۷۲ اتم سوپرسل** حساب می‌کنیم (نه فقط ۸ اتم سلول واحد)،
با همان منطق: موقعیت کارتزی هر اتم سوپرسل را با چرخش `R_cart` (حول مرکز سلول مرجع) و سپس
انتقال `t` تبدیل می‌کنیم، و نزدیک‌ترین اتم سوپرسل با همان عنصر را به‌عنوان متناظرش پیدا
می‌کنیم. سپس `IFC_new[i,j] = R_cart @ IFC[perm[i], perm[j]] @ R_cart^T` با `perm` کامل
۷۲تایی (که شامل permutation ۸تایی محور مرجع هم می‌شود، چون آن زیرمجموعه‌ای از سوپرسل است).

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("Installed")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS
import spglib

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
test_formula = common[0]
print(f"Test material: {test_formula}")

## بازسازی ماده با موتور واقعی Phonopy

In [ ]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


with open(band_yaml_files[test_formula]) as f:
    data = yaml.safe_load(f)

lattice = np.array(data['lattice'])
symbols = [p['symbol'] for p in data['points']]
frac_coords = np.array([p['coordinates'] for p in data['points']])
n_atoms = len(symbols)

unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

fc = parse_FORCE_CONSTANTS(str(fc_files[test_formula]))
n_supercell = fc.shape[1]

segment_len = data['segment_nqpoint'][0]
q_idx = min(segment_len // 2, len(data['phonon']) - 1)
q_test = data['phonon'][q_idx]['q-position']
real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n_supercell)
print(f"Discovered supercell_dim: {dim} (match error: {err:.6f})")

phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
phonon.force_constants = fc

phonon.run_qpoints([[0, 0, 0]])
original_freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
print(f"Original frequencies at Gamma (THz): {original_freqs}")

## کشف عملیات تقارن با spglib

In [ ]:
cell_tuple = (unitcell.cell, unitcell.scaled_positions, unitcell.numbers)
dataset = spglib.get_symmetry_dataset(cell_tuple, symprec=1e-3)

n_ops = len(dataset.rotations)
print(f"Space group: {dataset.international} (number {dataset.number})")
print(f"Number of symmetry operations: {n_ops}")

## تابع اصلاح‌شده: permutation کامل روی تمام اتم‌های سوپرسل (نه فقط سلول مرجع)

موقعیت کارتزی هر اتم سوپرسل را با `(R_cart, t_cart)` تبدیل می‌کنیم و نزدیک‌ترین اتم سوپرسل
معادل (همان عنصر، با در نظر گرفتن تصاویر دوره‌ای لاتیس سوپرسل) را پیدا می‌کنیم.

In [ ]:
def find_supercell_permutation(supercell, R_cart, t_cart, symprec=1e-3):
    """
    Apply the cartesian symmetry operation (R_cart, t_cart) to every supercell atom
    position and find the matching original supercell atom (same element, same
    position modulo the supercell lattice). Returns a permutation array of length
    n_supercell, or None if no valid bijection exists.
    """
    sc_cart = supercell.positions          # (n_supercell, 3)
    sc_symbols = supercell.symbols
    sc_lattice = supercell.cell            # (3,3), rows are lattice vectors
    inv_lattice = np.linalg.inv(sc_lattice)

    n = len(sc_cart)
    transformed = sc_cart @ R_cart.T + t_cart  # (n, 3) cartesian

    perm = np.full(n, -1, dtype=int)
    for i in range(n):
        best_j, best_dist = -1, np.inf
        for j in range(n):
            if sc_symbols[j] != sc_symbols[i]:
                continue
            diff_cart = transformed[i] - sc_cart[j]
            diff_frac = diff_cart @ inv_lattice
            diff_frac = diff_frac - np.round(diff_frac)  # wrap to nearest periodic image
            diff_cart_wrapped = diff_frac @ sc_lattice
            dist = np.linalg.norm(diff_cart_wrapped)
            if dist < best_dist:
                best_dist = dist
                best_j = j
        if best_dist > symprec:
            return None
        perm[i] = best_j

    if len(set(perm.tolist())) != n:
        return None
    return perm


def cartesian_rotation(R_frac, lattice):
    """Convert a fractional-coordinate rotation matrix to cartesian: R_cart = L^T @ R_frac @ L^-T"""
    L = lattice
    return L.T @ R_frac @ np.linalg.inv(L.T)


def cartesian_translation(t_frac, lattice):
    """Convert a fractional translation vector to cartesian."""
    return t_frac @ lattice

print("find_supercell_permutation ready")

In [ ]:
# find the first non-identity operation that gives a valid full-supercell permutation
test_op_idx = None
full_perm = None
for op_idx in range(1, n_ops):
    R = dataset.rotations[op_idx]
    t = dataset.translations[op_idx]
    R_cart = cartesian_rotation(R, lattice)
    t_cart = cartesian_translation(t, lattice)
    perm_candidate = find_supercell_permutation(phonon.supercell, R_cart, t_cart)
    if perm_candidate is not None:
        test_op_idx = op_idx
        full_perm = perm_candidate
        break

print(f"Testing with symmetry operation index: {test_op_idx}")
R = dataset.rotations[test_op_idx]
t = dataset.translations[test_op_idx]
R_cart = cartesian_rotation(R, lattice)
t_cart = cartesian_translation(t, lattice)
print(f"Rotation matrix (fractional):\n{R}")
print(f"Cartesian rotation matrix:\n{R_cart}")
print(f"Determinant: {np.linalg.det(R_cart):.4f}")
print(f"\nFull supercell permutation shape: {full_perm.shape} (should be ({n_supercell},))")
print(f"First 10 permutation values: {full_perm[:10]}")

## Transform کامل: permute هم محور مرجع و هم محور سوپرسل

این‌بار IFC را با `perm_unitcell` (زیرمجموعه‌ی ۸تایی اول از `full_perm`، چون اتم‌های سلول
واحد همان ۸ اتم اول سوپرسل‌اند) برای محور اول، و با `full_perm` کامل برای محور دوم
permute می‌کنیم.

In [ ]:
def transform_ifc_full(fc, perm_unitcell, perm_supercell, R_cart):
    """
    Transform the full IFC tensor (n_atoms, n_supercell, 3, 3) under a symmetry operation,
    permuting BOTH axes (reference unit-cell atoms and supercell image atoms).
    """
    fc_permuted = fc[perm_unitcell][:, perm_supercell]  # reorder both axes
    fc_new = np.einsum('ab,nscd,db->nsac', R_cart, fc_permuted, R_cart)
    return fc_new


# the unit-cell atoms are, by construction, the first n_atoms entries of the supercell
perm_unitcell = full_perm[:n_atoms]
print(f"Unit-cell sub-permutation: {perm_unitcell} (values should be < {n_atoms})")
assert np.all(perm_unitcell < n_atoms), "Unit-cell atoms did not map to other unit-cell atoms!"

fc_transformed = transform_ifc_full(fc, perm_unitcell, full_perm, R_cart)
print(f"\nTransformed IFC shape: {fc_transformed.shape} (should match original: {fc.shape})")

phonon_transformed = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
phonon_transformed.force_constants = fc_transformed
phonon_transformed.run_qpoints([[0, 0, 0]])
transformed_freqs = np.sort(phonon_transformed.get_qpoints_dict()['frequencies'][0])

print(f"\nOriginal frequencies at Gamma (THz):    {original_freqs}")
print(f"Transformed frequencies at Gamma (THz): {transformed_freqs}")
print(f"\nMax absolute difference: {np.max(np.abs(original_freqs - transformed_freqs)):.8f} THz")

## بررسی روی همه‌ی ۲۴ عملیات

اگر سلول قبلی موفق بود، این‌بار همه‌ی ۲۴ عملیات (نه فقط یکی) را تست می‌کنیم تا مطمئن شویم
نتیجه به‌طور سیستماتیک درست است، نه یک تصادف خوش‌شانس.

In [ ]:
max_errors = []

for op_idx in range(n_ops):
    R = dataset.rotations[op_idx]
    t = dataset.translations[op_idx]
    R_cart = cartesian_rotation(R, lattice)
    t_cart = cartesian_translation(t, lattice)

    full_perm_i = find_supercell_permutation(phonon.supercell, R_cart, t_cart)
    if full_perm_i is None:
        print(f"  op {op_idx}: FAILED to find a valid supercell permutation")
        continue

    perm_unitcell_i = full_perm_i[:n_atoms]
    if not np.all(perm_unitcell_i < n_atoms):
        print(f"  op {op_idx}: unit-cell atoms mapped outside the unit-cell block, skipping")
        continue

    fc_t = transform_ifc_full(fc, perm_unitcell_i, full_perm_i, R_cart)
    phonon_t = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon_t.force_constants = fc_t
    phonon_t.run_qpoints([[0, 0, 0]])
    freqs_t = np.sort(phonon_t.get_qpoints_dict()['frequencies'][0])

    max_err = np.max(np.abs(original_freqs - freqs_t))
    max_errors.append(max_err)
    status = "OK" if max_err < 1e-4 else "MISMATCH"
    print(f"  op {op_idx}: max frequency error = {max_err:.8f} THz [{status}]")

print(f"\nSummary: {len(max_errors)} / {n_ops} operations tested successfully")
print(f"Worst error across all tested operations: {max(max_errors):.8f} THz" if max_errors else "No operations tested")

## نتیجه‌گیری این تست

اگر این‌بار همه‌ی ۲۴ عملیات (یا اکثرشان) خطای فرکانس نزدیک صفر (کمتر از `1e-4`) داشته
باشند، یعنی فرمول transform کامل (با permutation هر دو محور IFC) درست است و می‌توانیم با
اطمینان این روش را روی کل دیتاست اعمال کنیم.

**لطفاً کل خروجی این نوت‌بوک را برای من بفرستید.**